# CatBoost (GPU, T4 x2) - minimal engineered features

Kaggle-GPU port of `Experiments.ipynb` keeping **only three** engineered features:
`contract_x_internet`, `contract_x_payment`, and `AverageMonthly`. CatBoost is tuned by a 60-trial Optuna study and refit with 5-fold CV
through the shared experiment harness.

**Both GPUs:** CatBoost trains each model across both T4s natively
(`task_type='GPU', devices='0:1'`), so every fit - study trials and the final refit -
uses both cores. (The LightGBM GPU notebooks instead ran one trial per GPU, since
LightGBM GPU is single-device.)

**Settings (right sidebar):** Accelerator -> GPU **T4 x2** (not P100); Internet -> On;
Add Input -> playground-series-s6e3.

> WARNING: Not verified against the live Kaggle environment. Run the GPU smoke-test cell
> and a ~4-trial study first before the full Save & Run All.

In [1]:
# List attached inputs (confirm the competition data is mounted).
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
import os, sys, subprocess

REPO_URL  = "https://github.com/biswajit-nag/Predict-Customer-Churn.git"
REPO_ROOT = "/kaggle/working/Predict-Customer-Churn"

if not os.path.exists(REPO_ROOT):
    subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)

os.chdir(REPO_ROOT)                       # CWD = repo root (fixes data paths + git_info)
# Put the clone FIRST on sys.path so its `src` wins over any other module named `src`,
# and drop a possibly-stale `src` cached by an earlier cell. (A plain
# `if REPO_ROOT not in sys.path` guard can leave a shadowing `src` ahead of ours.)
sys.path.insert(0, REPO_ROOT)
for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]
print("CWD:", os.getcwd())

Cloning into '/kaggle/working/Predict-Customer-Churn'...


CWD: /kaggle/working/Predict-Customer-Churn


Updating files: 100% (289/289), done.


In [3]:
# CatBoost's PyPI wheel ships with GPU support built in (unlike LightGBM, which needs a
# source build), and Kaggle's image usually already has it. Pin both to be safe.
!pip install -q catboost optuna

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())     # expect 2 on T4 x2

CUDA available: True
GPUs: 2


In [4]:
# cuda.is_available() can return True on an incompatible GPU - run a real op, then
# confirm CatBoost's GPU path fits on each device and across both (the study trains
# with devices='0:1', so the multi-GPU combo must work).
import numpy as np
from catboost import CatBoostClassifier

print("device:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())
try:
    _ = (torch.randn(16, device="cuda") @ torch.randn(16, 16, device="cuda")).sum().item()
    print("GPU compute OK")
except Exception as e:
    print("GPU compute FAILED:", e)            # if this fails, switch to T4 and restart

_Xs = np.random.rand(2000, 6)
_ys = (np.random.rand(2000) > 0.5).astype(int)
_devs = ['0'] if torch.cuda.device_count() < 2 else ['0', '1', '0:1']
for _d in _devs:
    try:
        CatBoostClassifier(iterations=10, task_type='GPU', devices=_d, verbose=0).fit(_Xs, _ys)
        print(f'CatBoost GPU OK on devices={_d}')
    except Exception as e:
        print(f'CatBoost GPU FAILED on devices={_d}:', e)

device: Tesla T4 | count: 2
GPU compute OK
CatBoost GPU OK on devices=0
CatBoost GPU OK on devices=1
CatBoost GPU OK on devices=0:1


In [5]:
# data/processed/*.parquet are git-ignored, so absent from the clone. Rebuild the
# NATIVE (category-dtype) frames from the attached competition CSVs - CatBoost takes the
# raw category strings via cat_features, so this is the encoding Experiments.ipynb uses.
import shutil
from pathlib import Path

raw_dir = Path(REPO_ROOT) / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
for f in ("train.csv", "test.csv"):
    shutil.copy(f"/kaggle/input/competitions/playground-series-s6e3/{f}", raw_dir / f)

from src.data import prepare_data
train_df, test_df = prepare_data(encoding='native', force=True)
print(f'Loaded native: train_df {train_df.shape}, test_df {test_df.shape}')

Preprocessed and saved (native): train_df (594194, 21), test_df (254655, 20)
Loaded native: train_df (594194, 21), test_df (254655, 20)


In [6]:
import json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

from src.tracking import DATA_DIR, RUNS_DIR, RUNS_CSV
from src.cv import run_cv_experiment, save_experiment

### Feature engineering

keeps contract_x_internet, contract_x_payment, AverageMonthly (3 engineered features)

In [7]:

# The '_native' suffix marks the base encoding (category-dtype categoricals, not
# one-hot) so this FE cache never collides with the one-hot parquets, and so a
# run's logged data_version unambiguously identifies which base it used. Bumped
# to fe_v3 when the docs/fe_ideas.md §1 high-cardinality crosses were added.
DATA_VERSION = 'fe_v4_native'  # bump whenever engineer_features changes — see rule below

# The service-profile string concatenates the 8 service flags + InternetService
# into one categorical (docs/fe_ideas.md §1).
SERVICE_COLS = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
                'InternetService']


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Row-wise (stateless) feature engineering.

    Each new column depends ONLY on values from the row being transformed — no
    aggregates, no fitted encoders, no cross-row statistics. Those leak val/test
    information into training when applied before the CV split, so they belong
    inside the fold loop instead.

    The high-cardinality crosses below are deliberately left as raw category
    strings here, because mapping a category to a number is the leakage-prone
    half and must be fitted per-fold inside the CV loop, never once up front.
    How that mapping happens depends on the model in the run-config cell:
    LightGBM splits on the categories natively (its per-fold tree build *is* the
    in-fold fit, so no separate encoder is needed), whereas an LR pipeline would
    target-encode them per fold instead. Either way the encoding is refit on each
    fold's training rows — leakage-safe by construction (docs/fe_ideas.md §1).
    """
    df = df.copy()

    df['AverageMonthly'] = df['TotalCharges'] / df['tenure']
    df['contract_x_payment'] = (df['Contract'].astype(str) + ' | '
                                + df['PaymentMethod'].astype(str)).astype('category')
    df['contract_x_internet'] = (df['Contract'].astype(str) + ' | '
                                 + df['InternetService'].astype(str)).astype('category')
    return df


# Cache engineered parquets per DATA_VERSION. Past runs are reconstructible by
# loading the parquet whose suffix matches that run's logged data_version.
fe_train_path = DATA_DIR / f'train_df_{DATA_VERSION}.parquet'
fe_test_path  = DATA_DIR / f'test_df_{DATA_VERSION}.parquet'

if fe_train_path.exists() and fe_test_path.exists():
    train_df = pd.read_parquet(fe_train_path)
    test_df  = pd.read_parquet(fe_test_path)
    print(f'Loaded cached FE: {DATA_VERSION}')
else:
    train_df = engineer_features(train_df)
    test_df  = engineer_features(test_df)
    pq.write_table(pa.Table.from_pandas(train_df, preserve_index=False), fe_train_path)
    pq.write_table(pa.Table.from_pandas(test_df,  preserve_index=False), fe_test_path)
    print(f'Computed and cached FE: {DATA_VERSION}')



Loaded cached FE: fe_v4_native


In [8]:
# Refresh feature list and design matrices from the engineered dataframes.
encoded_features = [c for c in train_df.columns if c not in ('id', 'Churn')]
X_train = train_df[encoded_features]
y_train = train_df['Churn']
X_test  = test_df[encoded_features]

# CatBoost needs the categorical columns named explicitly. Under native encoding the
# original categoricals AND the engineered crosses are all pandas `category` dtype.
cat_features = [c for c in encoded_features if str(X_train[c].dtype) == 'category']
print(f'X_train: {X_train.shape}  features: {len(encoded_features)}  categorical: {len(cat_features)}')

X_train: (594194, 22)  features: 22  categorical: 17


### Optuna study - CatBoost on T4 x2

60 trials, 3-fold inner CV, ROC-AUC objective. Each fit trains across both T4 GPUs (`devices='0:1'`), so trials run sequentially. Pin `run_config['params']` to `cat_study.best_params`, so run this cell before the run cell.

In [9]:
# --- Optuna CatBoost study on BOTH T4 GPUs ---
# CatBoost trains each model across both T4s natively (devices='0:1'), so trials run
# SEQUENTIALLY (n_jobs left at 1) - unlike the LightGBM GPU notebooks, which ran one trial
# per GPU because LightGBM GPU is single-device. Both approaches saturate the two T4s; this
# is the CatBoost-idiomatic, thread-safe one (concurrent CatBoost GPU fits are unsafe).
#
# CatBoost-appropriate search space (the LightGBM knobs - num_leaves, cat_smooth, ... - do
# not transfer). min_data_in_leaf is omitted: it needs a non-symmetric grow_policy on GPU.
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

N_GPUS    = max(1, min(2, torch.cuda.device_count()))
CAT_GPU   = dict(task_type='GPU', devices=('0:1' if N_GPUS >= 2 else '0'))
CAT_FIXED = dict(verbose=0, random_seed=42)


def catboost_objective(trial):
    params = {
        'iterations':          trial.suggest_int('iterations', 400, 1500),
        'learning_rate':       trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth':               trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg', 1.0, 30.0, log=True),
        'random_strength':     trial.suggest_float('random_strength', 0.1, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'border_count':        trial.suggest_int('border_count', 32, 255),
        **CAT_GPU,
        **CAT_FIXED,
    }
    # Manual fold loop instead of cross_val_score: cross_val_score calls sklearn.clone()
    # per fold, which round-trips constructor params and asserts each is the SAME object
    # via get_params(). CatBoostClassifier normalizes `cat_features` internally, so
    # get_params() hands back a different object and clone() raises "constructor ...
    # modifies parameter cat_features". Building a fresh model per fold (as src/cv.py does)
    # never calls clone(), so the check can't trip - and cat_features stays explicit.
    fold_aucs = []
    for tr_idx, va_idx in inner_cv.split(X_train, y_train):
        model = CatBoostClassifier(**params, cat_features=cat_features)
        model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        va_proba = model.predict_proba(X_train.iloc[va_idx])[:, 1]
        fold_aucs.append(roc_auc_score(y_train.iloc[va_idx], va_proba))
    return float(np.mean(fold_aucs))


cat_study = optuna.create_study(
    study_name='catboost-gpu',
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)

# SMOKE-TEST FIRST: drop n_trials to ~4 to confirm both GPUs light up (watch nvidia-smi)
# and check per-trial time before the full 60-trial Save & Run All.
cat_study.optimize(catboost_objective, n_trials=60, show_progress_bar=True)

print(f'Best inner-CV ROC AUC: {cat_study.best_value:.6f}  (trial {cat_study.best_trial.number})')
for k, v in cat_study.best_params.items():
    print(f'  {k:20s} {v}')

  0%|          | 0/60 [00:00<?, ?it/s]

Best inner-CV ROC AUC: 0.916370  (trial 54)
  iterations           1247
  learning_rate        0.09680496773520687
  depth                6
  l2_leaf_reg          1.293774973501806
  random_strength      0.10812232374192264
  bagging_temperature  0.09787890533355237
  border_count         245


### Run configuration

In [10]:
# --- Run config ---
# CatBoost on the engineered native features. best_params carries only the tuned
# hyperparameters; the GPU placement (CAT_GPU), fixed flags (CAT_FIXED) and cat_features
# are merged back in by the model factory. Every fit - study trials and the final 5-fold
# refit (src/cv.py, folds sequential) - trains across both T4s via devices='0:1'.
from catboost import CatBoostClassifier

X_test = test_df[encoded_features]

run_config = {
    'model_factory': lambda params: CatBoostClassifier(**params, **CAT_GPU, **CAT_FIXED,
                                                       cat_features=cat_features),
    'params':        cat_study.best_params,
    'metric':        accuracy_score,
    'metric_name':   'accuracy',
    'cv':            StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'tag':           'catboost-gpu-fe-min3',
    'notes':         "CatBoost (task_type=GPU) on Kaggle T4 x2, minimal FE: contract_x_internet + contract_x_payment + AverageMonthly only. Best params from a 60-trial Optuna study (3-fold inner CV, ROC-AUC); each fit trains across both T4s via devices='0:1'. Data regenerated on-platform; data_hash differs from local runs; GPU run not bit-reproducible. Notebook: kaggle/predict-customer-churn-catboost-gpu-min3.ipynb.",
    'parent_run_id': '',
    'save_models':   False,
    'data_version':  DATA_VERSION,
}

In [11]:
# Step 1 - Run the experiment (fits 5 folds, prints OOF accuracy + ROC-AUC).
result = run_cv_experiment(run_config, X_train, y_train, X_test, encoded_features)

Run ID: 20260611-051031-5d6bc5
Tag:    catboost-gpu-fe-min3

Fold 0: accuracy=0.8613  roc_auc=0.9161  (fit 59.8s)
Fold 1: accuracy=0.8612  roc_auc=0.9172  (fit 59.7s)
Fold 2: accuracy=0.8621  roc_auc=0.9166  (fit 60.4s)
Fold 3: accuracy=0.8631  roc_auc=0.9177  (fit 60.0s)
Fold 4: accuracy=0.8612  roc_auc=0.9149  (fit 60.4s)

OOF accuracy: 0.8618
OOF ROC-AUC:  0.9165
Folds:        0.8618 ± 0.0007

Run complete. Call save_experiment(result) to log this run permanently.


In [12]:
# Step 2 - Save the run (review the OOF ROC-AUC above first).
run_id = save_experiment(result)

Saved to: /kaggle/working/Predict-Customer-Churn/experiments/runs/20260611-051031-5d6bc5


### Build a submission (optional)

`test_proba_mean` is the fold-bagged churn probability for the full test set. The competition metric is ROC-AUC, so submit the probability directly.

In [13]:
submission = pd.DataFrame({
    'id':    test_df['id'],
    'Churn': result['artifacts']['test_proba_mean'],
})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.head())
print('wrote /kaggle/working/submission.csv', submission.shape)

       id     Churn
0  594194  0.073783
1  594195  0.001235
2  594196  0.100655
3  594197  0.003688
4  594198  0.498622
wrote /kaggle/working/submission.csv (254655, 2)


### Bundle run artifacts into one zip

Zips the run directory and `runs.csv` into a single archive on the Output tab. The source notebook is already on the Output tab (Kaggle saves it there), so it is not duplicated into the bundle.

In [14]:
import shutil
from pathlib import Path
from src.tracking import RUNS_DIR, RUNS_CSV

BUNDLE = Path('/kaggle/working/bundle')
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)

# 1) heavy run artifacts (params, oof_proba, test_proba_*, metrics, env, git diff)
shutil.copytree(RUNS_DIR / run_id, BUNDLE / 'runs' / run_id)
# 2) the master index row
shutil.copy(RUNS_CSV, BUNDLE / 'runs.csv')

archive = shutil.make_archive(f'/kaggle/working/{run_id}_bundle', 'zip', BUNDLE)
print('wrote', archive)

wrote /kaggle/working/20260611-051031-5d6bc5_bundle.zip
